# 2.02 贝尔曼方程：如何评价一个策略

[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ROI-su7/vla-ladder/blob/main/notebooks/02_rl/2.02_bellman_equation.ipynb)


## 为什么要学

要改进策略，首先得能**评价**策略：从每个状态出发，按这个策略走下去，平均能拿到多少回报？这个量叫**状态值**（state value）。它满足一个优美的方程，即贝尔曼方程。

后面所有 RL 算法（TD、Q-learning、DQN、Actor-Critic、PPO 里的 critic、RLT 的 Q 函数、RECAP 的价值函数）**本质上都是在用各种方式近似求解贝尔曼方程**。对应书中第 2 章。

---

## 1. 状态值

$$
v_\pi(s) = \mathbb{E}\left[G_t \mid S_t = s\right]
$$

从状态 $s$ 出发、按策略 $\pi$ 行动的**期望回报**。注意它依赖于策略 $\pi$：同一个格子，好策略下值高，差策略下值低。

> 如果环境和策略都是确定性的，那么期望就等于唯一一条轨迹的回报。

## 2. 推导贝尔曼方程

利用 $G_t = r_{t+1} + \gamma G_{t+1}$：

$$
v_\pi(s) = \mathbb{E}[R_{t+1}\mid S_t=s] + \gamma\,\mathbb{E}[G_{t+1}\mid S_t=s]
$$

第一项是**即时奖励的期望**，第二项是**未来回报的期望**。分别展开：

$$
\mathbb{E}[R_{t+1}\mid s] = \sum_a\pi(a|s)\sum_r p(r|s,a)\,r
$$

$$
\mathbb{E}[G_{t+1}\mid s] = \sum_a\pi(a|s)\sum_{s'}p(s'|s,a)\,v_\pi(s')\qquad\text{（马尔可夫性）}
$$

合起来就是**贝尔曼方程**：

$$
\boxed{v_\pi(s) = \sum_a\pi(a|s)\left[\sum_r p(r|s,a)\,r + \gamma\sum_{s'}p(s'|s,a)\,v_\pi(s')\right],\quad\forall s\in\mathcal{S}}
$$

它说的是：**一个状态的值 = 即时奖励 + γ × 下一个状态的值（都按概率取平均）**。所有状态的值互相依赖，共 $|\mathcal{S}|$ 个方程、$|\mathcal{S}|$ 个未知数。

## 3. 矩阵-向量形式

定义

- $r_\pi(s) = \sum_a\pi(a|s)\sum_r p(r|s,a)r$：在策略 $\pi$ 下状态 $s$ 的期望即时奖励；
- $P_\pi(s, s') = \sum_a\pi(a|s)p(s'|s,a)$：在策略 $\pi$ 下从 $s$ 到 $s'$ 的转移概率。

贝尔曼方程写成：

$$
\mathbf{v}_\pi = \mathbf{r}_\pi + \gamma P_\pi\mathbf{v}_\pi
$$

这就是 1.03 课预告的**线性方程组**！有两种解法：

**闭式解**：$\mathbf{v}_\pi = (I-\gamma P_\pi)^{-1}\mathbf{r}_\pi$。$P_\pi$ 每行和为 1、元素非负，所以它的特征值绝对值 $\le 1$，于是 $I-\gamma P_\pi$ 一定可逆。

**迭代解**：从任意 $\mathbf{v}_0$ 开始，反复迭代

$$
\mathbf{v}_{k+1} = \mathbf{r}_\pi + \gamma P_\pi\mathbf{v}_k
$$

会收敛到 $\mathbf{v}_\pi$。误差每次至少缩小 $\gamma$ 倍：$\|\mathbf{v}_{k}-\mathbf{v}_\pi\|_\infty\le\gamma^k\|\mathbf{v}_0-\mathbf{v}_\pi\|_\infty$。实际中状态多时用迭代解，这也是后面各种算法的基础。

---

## 4. 代码实现

In [ ]:
# 🔧 环境准备（每个 notebook 都先运行这一格）
# - 在 Colab 中：自动下载本项目代码
# - 在本地：自动把项目根目录加入 sys.path（也可以在根目录执行 `pip install -e .`）
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/vla-ladder"):
        os.system("git clone -q --depth 1 https://github.com/ROI-su7/vla-ladder.git /content/vla-ladder")
    sys.path.insert(0, "/content/vla-ladder")
else:
    _p = os.path.abspath(".")
    while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "vla_ladder")):
        _p = os.path.dirname(_p)
    sys.path.insert(0, _p)
print("环境就绪 ✅")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from vla_ladder.envs import GridWorld, policy_to_matrix

env = GridWorld()
gamma = 0.9

### 练习 1：计算 $\mathbf{r}_\pi$ 与 $P_\pi$

环境模型：`env.P` 形状 `(S, A, S')`，`env.R` 形状 `(S, A)`（期望即时奖励 $\sum_r p(r|s,a)r$）。策略 `pi` 形状 `(S, A)`。

In [ ]:
def policy_model(env, pi):
    """返回 r_pi (S,) 与 P_pi (S, S)"""
    # TODO: r_pi[s] = Σ_a pi[s,a] R[s,a]；P_pi[s,s'] = Σ_a pi[s,a] P[s,a,s']（用 einsum 很方便）
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def policy_model(env, pi):
    """返回 r_pi (S,) 与 P_pi (S, S)"""
    r_pi = (pi * env.R).sum(axis=1)
    P_pi = np.einsum("sa,sat->st", pi, env.P)
    return r_pi, P_pi

In [ ]:
# ✅ 自测
pi_rand = np.full((25, 5), 0.2)
r_pi, P_pi = policy_model(env, pi_rand)
assert r_pi.shape == (25,) and P_pi.shape == (25, 25)
assert np.allclose(P_pi.sum(1), 1)              # 每行是一个概率分布
assert np.isclose(r_pi[0], 0.2 * (-1 - 1))       # (0,0)：向上、向左撞墙各 -1
print("通过 ✅")

### 练习 2：闭式解与迭代解

In [ ]:
def evaluate_closed_form(env, pi, gamma):
    # TODO: 用 np.linalg.solve 解 (I - γ P_pi) v = r_pi
    raise NotImplementedError

def evaluate_iterative(env, pi, gamma, tol=1e-8, max_iter=10000):
    """返回 (v, 迭代次数)"""
    # TODO: v = 0；反复 v_new = r_pi + γ P_pi v，直到 max|v_new - v| < tol
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def evaluate_closed_form(env, pi, gamma):
    r_pi, P_pi = policy_model(env, pi)
    return np.linalg.solve(np.eye(env.n_states) - gamma * P_pi, r_pi)

def evaluate_iterative(env, pi, gamma, tol=1e-8, max_iter=10000):
    """返回 (v, 迭代次数)"""
    r_pi, P_pi = policy_model(env, pi)
    v = np.zeros(env.n_states)
    for k in range(max_iter):
        v_new = r_pi + gamma * P_pi @ v
        if np.max(np.abs(v_new - v)) < tol:
            return v_new, k + 1
        v = v_new
    return v, max_iter

In [ ]:
# ✅ 自测：两种方法结果一致
good = np.array([1, 1, 1, 1, 2, 0, 0, 1, 1, 2, 0, 3, 2, 1, 2, 0, 1, 4, 3, 2, 0, 1, 0, 3, 3])
pi_good = policy_to_matrix(good)
v1 = evaluate_closed_form(env, pi_good, gamma)
v2, iters = evaluate_iterative(env, pi_good, gamma)
assert np.allclose(v1, v2, atol=1e-6)
assert np.isclose(v1[env.to_index(env.target)], 1 / (1 - gamma))   # 目标格一直不动：1+γ+γ²+... = 10
print(f"通过 ✅ 迭代 {iters} 次收敛")

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
env.plot(V=v1, policy=pi_good, title="v_pi of the good policy", ax=ax[0])
env.plot(V=evaluate_closed_form(env, pi_rand, gamma), policy=pi_rand, title="v_pi of the random policy", ax=ax[1])
plt.show()

观察：

- 好策略下，离目标越近，值越高（$\gamma^k\cdot 10$ 的形式）；
- 随机策略下，几乎所有状态的值都是负的（经常撞墙、进禁区）；
- **状态值可以用来比较策略**：如果 $v_{\pi_1}(s)\ge v_{\pi_2}(s)$ 对所有 $s$ 成立，就说 $\pi_1$ 比 $\pi_2$ 好。

### 用蒙特卡洛验证：值 = 平均回报

状态值的定义是"期望回报"，那么多采样几条轨迹求平均，应该接近闭式解。这正是下一阶段"无模型"方法的出发点。

In [ ]:
rng = np.random.default_rng(0)
s0 = env.to_index((0, 0))
returns = []
for _ in range(2000):
    s = env.reset(s0); G, disc = 0.0, 1.0
    for t in range(100):                     # γ^100 ≈ 2.7e-5，截断误差可忽略
        a = rng.choice(5, p=pi_rand[s]); s, r = env.step(a)
        G += disc * r; disc *= gamma
    returns.append(G)
print(f"随机策略 v(0,0)：闭式解 {evaluate_closed_form(env, pi_rand, gamma)[s0]:.3f}，"
      f"2000 条轨迹平均回报 {np.mean(returns):.3f} ± {np.std(returns) / np.sqrt(2000):.3f}")

---

## 5. 动作值（action value）

$$
q_\pi(s,a) = \mathbb{E}[G_t\mid S_t=s, A_t=a]
$$

从 $s$ 出发、**先执行动作 $a$**、之后再按 $\pi$ 走的期望回报。它和状态值的关系：

$$
v_\pi(s) = \sum_a\pi(a|s)\,q_\pi(s,a),\qquad
q_\pi(s,a) = \sum_r p(r|s,a)\,r + \gamma\sum_{s'}p(s'|s,a)\,v_\pi(s')
$$

**为什么动作值更重要？** 因为它直接告诉我们"在这个状态下哪个动作更好"：只要选 $q$ 最大的动作，就能改进策略（下一课）。Q-learning、DQN、SAC、RLT 的 critic 学的都是 $q$。

### 练习 3：由状态值计算动作值

In [ ]:
def q_from_v(env, v, gamma):
    """v: (S,) -> q: (S, A)"""
    # TODO: q[s,a] = R[s,a] + γ Σ_s' P[s,a,s'] v[s']
    raise NotImplementedError

In [ ]:
#@title 💡 参考答案（点击展开查看；运行本单元即可使用参考实现） { display-mode: "form" }
def q_from_v(env, v, gamma):
    """v: (S,) -> q: (S, A)"""
    return env.R + gamma * env.P @ v      # (S,A,S') @ (S',) -> (S,A)

In [ ]:
# ✅ 自测
q = q_from_v(env, v1, gamma)
assert np.allclose((pi_good * q).sum(1), v1)        # v = Σ_a π(a|s) q(s,a)
s = env.to_index((0, 0))
print("在 (0,0) 处各动作的 q 值 [↑ → ↓ ← ○]:", q[s].round(2))
better = np.nonzero(q.argmax(1) != good)[0]
print("当前策略在这些格子里选的不是 q 最大的动作:", [env.to_rc(x) for x in better])
print("通过 ✅ —— 在这些格子改选 q 最大的动作，策略就会变好：这就是下一课的'策略改进'！")

---

## 小结

| 公式 | 含义 |
|---|---|
| $v_\pi(s)=\mathbb{E}[G_t\mid S_t=s]$ | 状态值：按 $\pi$ 走的期望回报 |
| $\mathbf{v}_\pi=\mathbf{r}_\pi+\gamma P_\pi\mathbf{v}_\pi$ | 贝尔曼方程（线性方程组） |
| $\mathbf{v}_\pi=(I-\gamma P_\pi)^{-1}\mathbf{r}_\pi$ | 闭式解 |
| $\mathbf{v}_{k+1}=\mathbf{r}_\pi+\gamma P_\pi\mathbf{v}_k$ | 迭代解（策略评估） |
| $q_\pi(s,a)=R(s,a)+\gamma\sum_{s'}p(s'\mid s,a)v_\pi(s')$ | 动作值 |

**思考题**：把 $r_{target}$ 从 1 改成 2，所有状态值会怎么变？如果把**所有**奖励都加上一个常数 $c$ 呢？最优策略会变吗？（书中第 3 章有完整讨论：所有奖励做仿射变换 $ar+b$，$a>0$，最优策略不变。）